# Final Project Phase 1: AI Agent-Powered Automation for Peloton's Fitness Ecosystem
# Shishir Deshpande, MSDS 442

### Importing required libraries

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv

load_dotenv() # this will read my secrets and API keys from the .env file
os.environ["USER_AGENT"] = "MSDS442-Assignment1-Deshpande"

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal
import fitz # this package has PyMuPDF tool
from IPython.display import display, Markdown

import pandas as pd

model = ChatOpenAI(model="gpt-4o-mini")

### Requirement 2

In [ ]:
# Setting up a prompt to send to ChatGPT (responsibilities copied from assignment instructions and requirements specification document)
prompt = """Peloton, the world’s largest interactive fitness platform, is known for its innovative approach to technology-enabled fitness and immersive, instructor-led classes. With a loyal community of over 6 million members, Peloton fosters a culture of fitness that is not only effective and convenient but also entertaining and socially engaging. From indoor cycling and running to yoga and meditation, Peloton’s classes/instructors guide members on their journey toward self-improvement and wellness.

I'm working on a requirement analysis project to build 5 multimodal LLM-based AI agents to help Peloton improve its business. For each of the 5 AI agents below, please generate 3 user stores (use cases). Each use case must be implementable with current LLM or agentic capabilities and each one recommendation should name a solid scenario where it can be used, what metric we can use to track, what product or situation it can apply to. It should also tie to responsibility listed below. The list of AI agents and their responsibilities is as follows: 

1. Business/Marketing AI Agent:
- Provide insights into ongoing marketing campaigns, promotions, and discounts.
- Answer inquiries about customer segmentation, lead generation, and campaign effectiveness.
- Assist employees with analytics on marketing ROI and engagement metrics.
- Suggest content strategies based on customer preferences and historical data.
- Resolve questions about brand performance or competitor analysis.

2. Data Science AI Agent:
- Deliver insights from fitness and performance data for both customers and employees.
- Provide answers about analytics tools, algorithms, or data trends used for decision-making.
- Respond to user queries about personalized fitness metrics or platform improvements.
- Explain data-driven recommendations or statistics to employees and customers.

3. Membership/Fraud Detection AI Agent:
- Address user concerns about suspicious account activity or unauthorized usage.
- Assist with password resets, account verification, and membership renewal questions.
- Guide customers through secure payment processes or dispute resolution.
- Answer inquiries related to member benefits, tier upgrades, or loyalty programs.

4. Order/Shipping AI Agent:
- Provide real-time updates on order status, shipping timelines, and delivery issues.
- Answer questions about returns, exchanges, or warranty policies.
- Assist employees with inventory management or fulfillment process queries.
- Resolve customer concerns about delayed or missing orders.
- Share shipping options, costs, and tracking details with customers or employees.

5. Product Recommendation AI Agent:
- Suggest personalized fitness equipment, apparel, or accessories based on user preferences.
- Answer questions about product features, compatibility, or pricing.
- Assist employees with cross-selling strategies or product bundling insights.
- Provide information about best-selling or trending items on the platform.
- Address customer concerns about product availability or recommendations

Here is an example format to use as a reference:
|AI Agent|Responsibility|User-Story|
|---|---|---|
|Business/Marketing AI Agent|Assist employees with analytics on marketing ROI and engagement metrics.|As a marketing specialist, I want the AI agent to analyze the success of the “Black Friday Peloton Tread Campaign” so that I can identify high-performing ads.|
|Data Science AI Agent|Deliver insights from fitness and performance data for both customers and employees.|As a Peloton member, I want the AI agent to review my last 30 indoor cycling classes and suggest ways to improve my average output and cadence.|
|Membership/Fraud Detection AI Agent|Address user concerns about suspicious account activity or unauthorized usage.|Who accessed my Peloton account from an unrecognized device, and how can I secure it immediately?|
|Order/Shipping AI Agent|Provide real-time updates on order status, shipping timelines, and delivery issues.|When will my Peloton Guide arrive, and what is its real-time delivery status?|
|Product Recommendation AI Agent|Suggest personalized fitness equipment, apparel, or accessories based on user preferences.|As a member, I want the AI agent to recommend Peloton-branded yoga mats and blocks based on my recent yoga class history so that I can enhance my practice.

Your response should be a tabulated output of 15 user stories (i.e., 3 per agent) in one table with the columns "AI Agent", "Responsibility", "User-Story". Please do not repeat the same responsibility or use case for one agent. 
"""

For reference:
|AI Agent|Responsibility|User-Story|
|---|---|---|
|Business/Marketing AI Agent|Assist employees with analytics on marketing ROI and engagement metrics.|As a marketing specialist, I want the AI agent to analyze the success of the “Black Friday Peloton Tread Campaign” so that I can identify high-performing ads.|
|Data Science AI Agent|Deliver insights from fitness and performance data for both customers and employees.|As a Peloton member, I want the AI agent to review my last 30 indoor cycling classes and suggest ways to improve my average output and cadence.|
|Membership/Fraud Detection AI Agent|Address user concerns about suspicious account activity or unauthorized usage.|Who accessed my Peloton account from an unrecognized device, and how can I secure it immediately?|
|Order/Shipping AI Agent|Provide real-time updates on order status, shipping timelines, and delivery issues.|When will my Peloton Guide arrive, and what is its real-time delivery status?|
|Product Recommendation AI Agent|Suggest personalized fitness equipment, apparel, or accessories based on user preferences.|As a member, I want the AI agent to recommend Peloton-branded yoga mats and blocks based on my recent yoga class history so that I can enhance my practice.|

In [ ]:
# Sending the prompt to ChatGPT to generate the user stories per Requirement 2
response = model.invoke([HumanMessage(content=prompt)])

# Store output to reuse in Requirement 3 (Create a Table for every AI Agent and every user-story (use-case)) and Requirement 4 (Use ChatGPT to generate training and testing data that you will use in your design, implementation, and testing of every AI agent and its user-stories you listed above.)
user_stories_output = response.content

display(Markdown(user_stories_output))

### Requirement 4

In [ ]:
# Requirement 4 is generating training and testing data for all 15 user stories
training_testing_prompt = f"""Based on the 15 AI agent user stories below, please generate training and testing data for each one of them. 

{user_stories_output}

For each user story, you will need to give: 
- 2 training example queries (realistic requests that could be sent by members or employees) and an expected response for each query. 
- 1 testing example query (to be effective, include at least one edge case that can trip up an AI) with its expected response

Your response should be a tabulated output with these columns: "AI Agent", "User-Story", "Data Type (Training/Testing)", "Sample Query", "Expected Response"
"""

response2 = model.invoke([HumanMessage(content = training_testing_prompt)])
training_testing_output = response2.content
display(Markdown(training_testing_output))

In [ ]:
### Using code to write it into a csv dataset
# Splitting into lines and keeping only table rows
lines = [l.strip() for l in training_testing_output.strip().split("\n") if l.strip().startswith("|")]
lines = [l for l in lines if "---" not in l]

# Converting each line into a list of cell values by splitting on "|" separator
rows = [[cell.strip() for cell in line.strip("|").split("|")] for line in lines]

# Fixing first row as header
training_testing_output_df = pd.DataFrame(rows[1:], columns = rows[0])

# Saving to CSV
training_testing_output_df.to_csv("Requirement 4_TrainingTestingData.csv", index = False)